# SABRE - Analytical model for a three-spin system
In this notebook, we derive an analytical model for the coherent polarization transfer using a three-spin system as the spin dynamical model. We will construct the Hamiltonian in a singlet-triplet Zeeman basis set, find out the LAC conditions, and calculate an example value for a system where the target nucleus is the $^{19}\mathrm{F}$ from an equatorial 3-fluoropyridine ligand.

We use SymPy and NumPy to perform the calculations.

In [ ]:
from sympy import *
from sympy.physics.quantum import TensorProduct
import numpy as np

### Define symbols
We define the angular frequencies $\omega_i$ for each spin $i$ and the scalar coupling constants $J_{ij}$ between each spin pair $ij$. In addition, we define symbols for the chemical shift $\delta_i$, gyromagnetic ratio $\gamma_i$, and the static polarization transfer field $B_0$.

In [ ]:
w1, w2, w3 = symbols(r'\omega_1, \omega_2, \omega_3', real=True)
J12, J13, J23 = symbols('J_12, J_13, J_23', real=True)
d1, d2, d3 = symbols(r'\delta_1, \delta_2, \delta_3', real=True)
y1, y2, y3 = symbols(r'\gamma_1, \gamma_2, \gamma_3', real=True)
B0 = symbols('B_0', real=True)

### Calculate spin operators
First, we define the one-spin operators for spin-1/2 nucleus:
$$
\hat{E} = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix},\
\hat{I}_x = \begin{bmatrix} 0 & 1/2 \\ 1/2 & 0 \end{bmatrix},\
\hat{I}_y = \begin{bmatrix} 0 & -i/2 \\ i/2 & 0 \end{bmatrix},\
\hat{I}_z = \begin{bmatrix} 1/2 & 0 \\ 0 & -1/2 \end{bmatrix}
$$
Then, we calculate all the necessary spin operators for the three-spin system using Kronecker products.

In [ ]:
Id = Matrix([[1, 0],
            [0, 1]])
Ix = Matrix([[0, Rational(1/2)],
            [Rational(1/2), 0]])
Iy = Matrix([[0, -I*Rational(1/2)],
            [I*Rational(1/2), 0]])
Iz = Matrix([[Rational(1/2), 0],
             [0, -Rational(1/2)]])

E = TensorProduct(Id, Id, Id)

Ix1 = TensorProduct(Ix, Id, Id)
Ix2 = TensorProduct(Id, Ix, Id)
Ix3 = TensorProduct(Id, Id, Ix)

Iy1 = TensorProduct(Iy, Id, Id)
Iy2 = TensorProduct(Id, Iy, Id)
Iy3 = TensorProduct(Id, Id, Iy)

Iz1 = TensorProduct(Iz, Id, Id)
Iz2 = TensorProduct(Id, Iz, Id)
Iz3 = TensorProduct(Id, Id, Iz)

I1I2 = Ix1*Ix2 + Iy1*Iy2 + Iz1*Iz2
I1I3 = Ix1*Ix3 + Iy1*Iy3 + Iz1*Iz3
I2I3 = Ix2*Ix3 + Iy2*Iy3 + Iz2*Iz3

### Define the Hamiltonian
The Hamiltonian for our spin system contains the Zeeman interaction, isotropic chemical shift, and scalar $J$-coupling. It is defined as follows:
$$
\hat{H} = \sum_{i}\omega_i\hat{I}_{iz} + \sum_{i<j}2\pi J_{ij} \hat{\mathbf{I}}_i \cdot \hat{\mathbf{I}}_j
$$

In [ ]:
H = w1*Iz1 + w2*Iz2 + w3*Iz3 + 2*pi*(J12*I1I2 + J13*I1I3 + J23*I2I3)

### Define the singlet-triplet Zeeman basis
Next, we define the basis set where the hydride protons are represented using the singlet-triplet states and the target nucleus is represented by the Zeeman states. The Zeeman states are:
$$
\ket{\alpha} = \begin{bmatrix} 1 \\ 0 \end{bmatrix},\ \ket{\beta} = \begin{bmatrix} 0 \\ 1 \end{bmatrix}
$$
The singlet-triplet states are:
\begin{align*}
\ket{S} &= \frac{1}{\sqrt{2}} \left( \ket{\alpha \beta} - \ket{\beta \alpha} \right) \\
\ket{T_+} &= \ket{\alpha \alpha} \\
\ket{T_0} &= \frac{1}{\sqrt{2}} \left( \ket{\alpha \beta} + \ket{\beta \alpha} \right) \\
\ket{T_-} &= \ket{\beta \beta}
\end{align*}
And the final basis set contains the following states:
\begin{align*}
M &= 1/2: & \left\{ \ket{S\alpha},\ \ket{T_+ \beta},\ \ket{T_0 \alpha} \right\} \\
M &= -1/2: & \left\{ \ket{S\beta},\ \ket{T_- \alpha},\ \ket{T_0 \beta} \right\} \\
M &= 1: & \left\{ \ket{T_+\alpha} \right\} \\
M &= -1: & \left\{ \ket{T_-\beta} \right\}
\end{align*}

In [ ]:
a = Matrix([1, 0])
b = Matrix([0, 1])

S = sqrt(Rational(1/2)) * (TensorProduct(a, b) - TensorProduct(b, a))
Tp = TensorProduct(a, a)
T0 = sqrt(Rational(1/2)) * (TensorProduct(a, b) + TensorProduct(b, a))
Tm = TensorProduct(b, b)

Sa = TensorProduct(S, a)
Tpb = TensorProduct(Tp, b)
T0a = TensorProduct(T0, a)

Sb = TensorProduct(S, b)
Tma = TensorProduct(Tm, a)
T0b = TensorProduct(T0, b)

Tpa = TensorProduct(Tp, a)

Tmb = TensorProduct(Tm, b)

### Perform the basis set transformation
We construct the change-of-basis matrix $U$ and perform the basis set tranformation:
$$
\tilde{H} = U^{-1}\hat{H}U
$$
The Hamiltonian becomes block-diagonal with two $3 \times 3$ blocks and two $1 \times 1$ blocks. The smaller blocks can be discarded. The $3 \times 3$ blocks are displayed for $M = 1/2$ and $M = -1/2$, respectively.

In [ ]:
U = Matrix.hstack(Sa, Tpb, T0a, Sb, Tma, T0b, Tpa, Tmb)

Ht = U.inv() * H * U
Ht = expand(Ht)

display("First block (M = 1/2)")
display(Ht[:3,:3])

display("Second block (M = -1/2)")
display(Ht[3:6, 3:6])

### Calculate the level anti-crossing
Level anti-crossings may occur between states $\ket{S_\alpha}$ and $\ket{T_p \beta}$, or $\ket{S_\beta}$ and $\ket{T_m \alpha}$ when their diagonal elements in the Hamiltonian become equal. This sets a matching condition for the $B_0$.

First, we substitute the effective Larmor frequencies $\omega_i = -\gamma_i B_0 (1 + \delta_i)$. Then, we solve for the $B_0$ that satisfies the LAC condition in both cases. Finally, we calculate an example value for a 3-fluoropyridine system, where the $J$-couplings are $J_{12} = -7.67\ \mathrm{Hz}$, $J_{13} = -0.05\ \mathrm{Hz}$, and $J_{23} = 2.08\ \mathrm{Hz}$, and the chemical shifts are $\delta_1 = \delta_2 = -23.1\ \mathrm{ppm}$ and $\delta_3 = -124.2\ \mathrm{ppm}$.

In [ ]:
Ht = Ht.subs([
    (w1, -y1 * B0 * (1 + d1)),
    (w2, -y2 * B0 * (1 + d2)),
    (w3, -y3 * B0 * (1 + d3))
])

eq_LAC1 = Ht[0,0] - Ht[1,1]
eq_LAC2 = Ht[3,3] - Ht[4,4]

B0_LAC1 = solve(eq_LAC1, B0)[0]
B0_LAC2 = solve(eq_LAC2, B0)[0]

display(B0_LAC1)
display(B0_LAC2)

B0_LAC2_example = B0_LAC2.subs([
    (J12, -7.67),
    (J13, -0.05),
    (J23, 2.08),
    (d1, -23.1*1e-6),
    (d2, -23.1*1e-6),
    (d3, -124.2*1e-6),
    (y1, 267522188),
    (y2, 267522188),
    (y3, 251762483),
    (pi, np.pi)
])
display(B0_LAC2_example)